# Sarcomere detection pipeline

Reproducible companion to the sarcomere-integrity supplemental figures.
For each dataset (ET-1, mavacamten) the pipeline (a) detects Z-discs from α-actinin-GFP fluorescence images, (b) computes the Pasqualini/Grosberg-framework metrics **SPD** (sarcomeric packing density, crinkle-robust lattice periodicity) and **γ** (fraction of foreground detected as elongated Z-disks rather than punctate Z-bodies), and (c) renders the paired-tissue figure used in the paper.

**How to use**

1. Update the paths and representative-tissue IDs in the `CONFIGS` dict below to match your data.
2. `Run All` (menu → Run → Run All Cells).
3. Figures land in `sarcomere_pipeline/figures/` and per-image metrics land in `sarcomere_pipeline/results/`.

**Metrics implemented (references)**

- Pasqualini, F. S. *et al.* Structural phenotyping of stem cell-derived cardiomyocytes. *Stem Cell Reports* **4**, 340–347 (2015).
- Sutcliffe, M. D. *et al.* High-content analysis of sarcomeric structure in hiPSC-CMs. *Nat. Commun.* **9**, 5296 (2018).
- Toepfer, C. N. *et al.* SarcTrack — automated analysis of sarcomere function in hiPSC-CMs. *Circulation Research* **124**, 1172–1183 (2019).

The detector portion (LoG + adaptive threshold + PCA per blob) is our own implementation, calibrated for α-actinin-GFP EHT strips at ~0.16 μm/pixel.

In [ ]:
# Install dependencies (safe to re-run; skipped on subsequent runs)
%pip install --quiet nd2 opencv-python matplotlib scipy scikit-image numpy

In [ ]:
# ============================================================
# Imports, output paths, and rendering defaults
# ============================================================
%matplotlib inline
import os, re, csv, math, time, json
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from numpy.fft import rfft, rfftfreq
from scipy import ndimage as ndi
from scipy import stats as sps

OUTPUT_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
# When run as a notebook __file__ is undefined; fall back to the current
# working directory. If you moved this notebook, set OUTPUT_DIR manually.
OUTPUT_DIR = Path("/Users/ersm/sarcomere_pipeline")
FIG_DIR    = OUTPUT_DIR / "figures"
RESULT_DIR = OUTPUT_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "figure.dpi": 120,
})

print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"  figures/  → {FIG_DIR}")
print(f"  results/  → {RESULT_DIR}")

In [ ]:
# ============================================================
# DETECTOR — Z-disc detection and fiber-axis estimation
# ============================================================
# Pipeline per image:
#   1. Load .nd2 / .tif / standard 2D image → single-channel gray
#   2. Background haze removal (morphological top-hat)
#   3. Otsu threshold on blurred haze-removed → tissue_mask
#   4. LoG + CLAHE + unsharp mask → sharpened
#   5. Adaptive + local top-hat thresholds → Z-disc candidate mask
#   6. Connected components → per-blob PCA (center, orientation, aspect,
#      long-axis length). Filter by area 3-5000 px and long axis ≤ 10 px.
#   7. Global fiber axis: max of structure-tensor eigenvalue vs. tensor
#      of per-Z-disc PCA vectors (`choose_fiber_axis`).
# All downstream metrics consume the `quantify_tissue` result.
# ============================================================

def px2um(v, um_per_px):
    return v * um_per_px

def um2px(v, um_per_px):
    return v / um_per_px

def load_image(path):
    """Return (bgr, gray) for a .nd2 / .tif / standard 2D image."""
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    ext = os.path.splitext(path)[1].lower()
    if ext == ".nd2":
        try:
            import nd2
        except ImportError:
            raise RuntimeError("`.nd2` file requested. Install: %pip install nd2")
        with nd2.ND2File(path) as f:
            arr = np.asarray(f.asarray())
        while arr.ndim > 2:
            arr = arr.max(axis=0)
        a = arr.astype(np.float32)
        a -= a.min()
        m = float(a.max())
        gray = ((a / m) * 255).astype(np.uint8) if m > 0 else a.astype(np.uint8)
        bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        return bgr, gray
    bgr = cv2.imread(path)
    if bgr is None:
        raise FileNotFoundError(f"Could not read {path}")
    return bgr, bgr[:, :, 1]

def sharpen(gray, log_sigma_px=1.5, unsharp_sigma=1.0, unsharp_amount=1.2):
    """LoG response → CLAHE → unsharp mask → clipped uint8."""
    g = gray.astype(np.float32)
    smoothed = cv2.GaussianBlur(g, (0, 0), log_sigma_px)
    log_response = -cv2.Laplacian(smoothed, cv2.CV_32F, ksize=3)
    log_response = np.clip(log_response, 0, None)
    if log_response.max() > 0:
        log_response = log_response * (255.0 / log_response.max())
    log_u8 = log_response.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16, 16))
    equalized = clahe.apply(log_u8)
    blurred = cv2.GaussianBlur(equalized.astype(np.float32), (0, 0), unsharp_sigma)
    out = np.clip(equalized + unsharp_amount * (equalized - blurred), 0, 255)
    pos = out[out > 5]
    if pos.size:
        lo, hi = np.percentile(pos, [1, 99.5])
        if hi > lo:
            out = np.clip((out - lo) * (255.0 / (hi - lo)), 0, 255)
    return out.astype(np.uint8)

def pca_orientation(points_xy):
    """Return (center, major_vec, major_len, aspect) for a set of 2D points."""
    points_xy = np.asarray(points_xy, dtype=np.float32)
    center = points_xy.mean(axis=0)
    if len(points_xy) < 3:
        return center, None, None, None
    centered = points_xy - center
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, order]
    major_vec = eigvecs[:, 0]
    major_proj = centered @ major_vec
    minor_proj = centered @ eigvecs[:, 1]
    major_len = np.percentile(major_proj, 95) - np.percentile(major_proj, 5)
    minor_len = np.percentile(minor_proj, 95) - np.percentile(minor_proj, 5)
    aspect = major_len / (minor_len + 1e-6)
    return center, major_vec, major_len, aspect

def detect_zdiscs(gray, min_aspect=1.15, max_disc_len_px=10):
    """Return (tissue_mask, clean_zdiscs, z_discs, sharpened, n_fused_rejected).
    z_discs is a list of dicts: center=[x,y], disc_vec=[dx,dy], disc_len (px),
    aspect, oriented (bool = aspect >= min_aspect)."""
    kernel_bg = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (45, 45))
    haze_removed = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel_bg)
    blur_tissue = cv2.GaussianBlur(haze_removed, (35, 35), 0)
    _, raw_mask = cv2.threshold(blur_tissue, 0, 255,
                                cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    tissue_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN, kernel_clean)

    sharpened = sharpen(haze_removed)

    thresh_a = cv2.adaptiveThreshold(sharpened, 255,
                                     cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 11, -5)
    small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    tophat_small = cv2.morphologyEx(sharpened, cv2.MORPH_TOPHAT, small)
    _, thresh_b = cv2.threshold(tophat_small, 10, 255, cv2.THRESH_BINARY)
    thresh_b = cv2.erode(thresh_b, np.ones((2, 2), np.uint8))
    combined = cv2.bitwise_or(thresh_a, thresh_b)
    combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN,
                                np.ones((2, 2), np.uint8))
    clean_zdiscs = cv2.bitwise_and(combined, combined, mask=tissue_mask)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        clean_zdiscs, connectivity=8)
    slices = ndi.find_objects(labels)

    z_discs = []
    fused_rejected = 0
    for lid in range(1, num_labels):
        area = int(stats[lid, cv2.CC_STAT_AREA])
        if area < 3 or area > 5000:
            continue
        sl = slices[lid - 1]
        if sl is None:
            continue
        sy, sx = sl
        sub = labels[sy, sx] == lid
        ys, xs = np.nonzero(sub)
        ys = ys + sy.start
        xs = xs + sx.start
        center, disc_vec, disc_len, aspect = pca_orientation(
            np.column_stack([xs, ys]))
        if disc_vec is None or disc_len < 2.5:
            continue
        if disc_len > max_disc_len_px:
            fused_rejected += 1
            continue
        z_discs.append({
            "center":   center,
            "disc_vec": disc_vec,
            "disc_len": disc_len,
            "aspect":   aspect,
            "oriented": aspect >= min_aspect,
        })
    return tissue_mask, clean_zdiscs, z_discs, sharpened, fused_rejected

def _angle_between_deg(u, v):
    cosang = abs(float(np.dot(u, v))) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9)
    return math.degrees(math.acos(np.clip(cosang, -1.0, 1.0)))

def estimate_fiber_axis(enhanced, tissue_mask):
    """Global fiber axis from a whole-image structure tensor."""
    img_f = enhanced.astype(np.float32)
    gx = cv2.Sobel(img_f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(img_f, cv2.CV_32F, 0, 1, ksize=3)
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    inner_mask = cv2.erode(tissue_mask, kern) > 0
    if inner_mask.sum() < 100:
        return np.array([1.0, 0.0], dtype=np.float32), 0.0
    Sxx = float((gx * gx)[inner_mask].sum())
    Syy = float((gy * gy)[inner_mask].sum())
    Sxy = float((gx * gy)[inner_mask].sum())
    S = np.array([[Sxx, Sxy], [Sxy, Syy]], dtype=np.float64)
    eigvals, eigvecs = np.linalg.eigh(S)
    fiber_vec = eigvecs[:, 1].astype(np.float32)
    coherence = float((eigvals[1] - eigvals[0]) / (eigvals[1] + eigvals[0] + 1e-9))
    if abs(fiber_vec[0]) >= abs(fiber_vec[1]):
        if fiber_vec[0] < 0: fiber_vec = -fiber_vec
    else:
        if fiber_vec[1] < 0: fiber_vec = -fiber_vec
    return fiber_vec, coherence

def zdisc_fiber_axis(z_discs):
    """Fiber axis from the tensor of per-Z-disc PCA vectors (fallback)."""
    oriented = [zd for zd in z_discs if zd["oriented"]]
    if len(oriented) < 5:
        return np.array([1.0, 0.0], dtype=np.float32), 0.0
    V = np.stack([zd["disc_vec"] for zd in oriented]).astype(np.float64)
    V /= (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    T = V.T @ V
    eigvals, eigvecs = np.linalg.eigh(T)
    disc_axis = eigvecs[:, -1]
    coherence = float((eigvals[-1] - eigvals[0]) / (eigvals[-1] + eigvals[0] + 1e-9))
    fiber_vec = np.array([-disc_axis[1], disc_axis[0]], dtype=np.float32)
    if abs(fiber_vec[0]) >= abs(fiber_vec[1]):
        if fiber_vec[0] < 0: fiber_vec = -fiber_vec
    else:
        if fiber_vec[1] < 0: fiber_vec = -fiber_vec
    return fiber_vec, coherence

def choose_fiber_axis(struct_vec, struct_coh, disc_vec, disc_coh,
                      disagreement_threshold_deg=30.0,
                      struct_conf_floor=0.15):
    """Reconcile the structure-tensor axis with the Z-disc axis."""
    delta = _angle_between_deg(struct_vec, disc_vec)
    if delta <= disagreement_threshold_deg:
        return struct_vec, max(struct_coh, disc_coh), "tensor"
    if struct_coh < struct_conf_floor and disc_coh > struct_coh:
        return disc_vec, disc_coh, "zdisc-override"
    if disc_coh > struct_coh + 0.1:
        return disc_vec, disc_coh, "zdisc-override"
    return struct_vec, min(struct_coh, disc_coh), "tensor (weak agreement)"

def quantify_tissue(image_path, um_per_px):
    """Run the detector once on one image. Returns a dict consumed by
    `paper_metrics`. See top-of-cell docstring for the full pipeline."""
    _, gray = load_image(image_path)
    tissue_mask, clean_zdiscs, z_discs, sharpened, _ = detect_zdiscs(gray)
    struct_vec, struct_coh = estimate_fiber_axis(sharpened, tissue_mask)
    disc_vec, disc_coh     = zdisc_fiber_axis(z_discs)
    fiber_vec, _fiber_coh, _axis_method = choose_fiber_axis(
        struct_vec, struct_coh, disc_vec, disc_coh)
    tissue_area_um2 = float((tissue_mask > 0).sum()) * (um_per_px ** 2)
    return {
        "path":            str(image_path),
        "gray":            gray,
        "sharpened":       sharpened,
        "tissue_mask":     tissue_mask,
        "clean_zdiscs":    clean_zdiscs,
        "z_discs":         z_discs,
        "fiber_vec":       tuple(map(float, fiber_vec)),
        "n_zdiscs":        len(z_discs),
        "tissue_area_um2": tissue_area_um2,
        "um_per_px":       um_per_px,
    }

print("detector module ready: quantify_tissue(path, um_per_px)")

In [ ]:
# ============================================================
# METRICS — Pasqualini/Grosberg-framework
#   OOP    : orientation order parameter from per-Z-disc PCA vectors
#            |<e^{i·2φ}>|  in [0, 1]. Random = 0, perfectly aligned = 1.
#   γ      : fraction of detected foreground blobs that are elongated
#            Z-disks (aspect ≥ 1.15) vs. punctate Z-bodies.
#   wOOP   : γ · OOP  — combines abundance-of-Z-disks with alignment.
#   SL_fft : sarcomere length from 2D radial FFT peak (µm).
#   SPD    : sarcomeric packing density — fraction of Z-disc mass
#            whose 1D FFT along the *local* fiber direction lands in
#            a ±20% band around 1/SL. Robust to bending/crinkling.
# All functions consume the `quantify_tissue` result dict.
# ============================================================

def _local_orientation_field(sharp, tissue_mask, sigma_px=25.0):
    """Structure-tensor local orientation, σ ≈ 4 μm (larger than a
    sarcomere) so we get the *fiber* direction not per-pixel gradients."""
    g = sharp.astype(np.float32)
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    Jxx = cv2.GaussianBlur(gx * gx, (0, 0), sigma_px)
    Jyy = cv2.GaussianBlur(gy * gy, (0, 0), sigma_px)
    Jxy = cv2.GaussianBlur(gx * gy, (0, 0), sigma_px)
    d = Jxx - Jyy
    disc = np.sqrt(d * d + 4 * Jxy * Jxy + 1e-12)
    lam_max = 0.5 * ((Jxx + Jyy) + disc)
    lam_min = 0.5 * ((Jxx + Jyy) - disc)
    theta = 0.5 * np.arctan2(2 * Jxy, d)
    coh = (lam_max - lam_min) / (lam_max + lam_min + 1e-9)
    mask = (tissue_mask > 0)
    return np.where(mask, theta, 0), np.where(mask, coh, 0)

def _sarcomere_length_um_fft(sharp, tissue_mask, um_per_px,
                             min_um=1.2, max_um=2.6):
    """Whole-image 2D radial FFT → dominant period between min_um and max_um.
    Sensitive to global tissue bending; use as a target frequency for SPD."""
    s = sharp.astype(np.float32).copy()
    s -= s.mean()
    s *= (tissue_mask > 0)
    H, W = s.shape
    F = np.fft.fftshift(np.abs(np.fft.fft2(s)))
    cy, cx = H // 2, W // 2
    y, x = np.indices((H, W))
    r = np.hypot(y - cy, x - cx).astype(int)
    tbin = np.bincount(r.ravel(), F.ravel())
    nr = np.bincount(r.ravel())
    with np.errstate(invalid='ignore', divide='ignore'):
        radial = tbin / np.maximum(nr, 1)
    freqs = np.arange(len(radial)) / max(H, W)
    with np.errstate(divide='ignore'):
        period_px = np.where(freqs > 0, 1.0 / freqs, np.inf)
    period_um = period_px * um_per_px
    valid = (period_um >= min_um) & (period_um <= max_um) & np.isfinite(period_um)
    if not valid.any():
        return float('nan')
    return float(period_um[np.argmax(radial * valid)])

def _oop_from_zdiscs(z_discs):
    """OOP = |<e^{i·2φ}>| over per-Z-disc PCA orientation vectors."""
    oriented = [zd for zd in z_discs if zd["oriented"]]
    if len(oriented) < 20:
        return float('nan'), 0
    V = np.stack([zd["disc_vec"] for zd in oriented]).astype(np.float64)
    V /= (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    phi = np.arctan2(V[:, 1], V[:, 0])
    c = np.cos(2 * phi).mean()
    s = np.sin(2 * phi).mean()
    return float(np.clip(np.hypot(c, s), 0.0, 1.0)), len(oriented)

def _gamma_zdisks_over_bodies(z_discs, min_aspect=1.15):
    """γ = n_elongated / n_total_detected. Punctate Z-bodies drop this."""
    n = len(z_discs)
    if n == 0:
        return float('nan')
    n_disk = sum(1 for zd in z_discs if zd.get("oriented", False))
    return float(n_disk / n)

def _spd_along_fibers_v2(sharp, clean_zdiscs, tissue_mask, theta_field,
                         SL_um, um_per_px,
                         patch_um=30.0, sl_tol_pct=0.25,
                         min_zdisc_frac=0.05):
    """Crinkle-robust lattice periodicity. Walks each patch along its
    LOCAL fiber direction, takes a 1D FFT, and asks what fraction of
    the spectral power sits in a ±sl_tol_pct band around 1/SL. Only
    patches with ≥ min_zdisc_frac Z-disc pixels count; each patch is
    weighted by its own Z-disc mass."""
    if not np.isfinite(SL_um) or SL_um <= 0:
        return float('nan')
    H, W = sharp.shape
    patch_px = max(8, int(patch_um / um_per_px))
    step = max(patch_px // 3, 4)
    target_freq = 1.0 / SL_um
    tol = sl_tol_pct * target_freq
    lo, hi = target_freq - tol, target_freq + tol
    n_target_pix = int(min_zdisc_frac * patch_px * patch_px)
    numer, denom, n_patches = 0.0, 0.0, 0
    for cy in range(patch_px // 2, H - patch_px // 2, step):
        for cx in range(patch_px // 2, W - patch_px // 2, step):
            y0, y1 = cy - patch_px // 2, cy + patch_px // 2
            x0, x1 = cx - patch_px // 2, cx + patch_px // 2
            tm = tissue_mask[y0:y1, x0:x1]
            if tm.sum() < 0.7 * tm.size:
                continue
            zm = clean_zdiscs[y0:y1, x0:x1]
            n_zpix = int((zm > 0).sum())
            if n_zpix < n_target_pix:
                continue
            th = np.median(theta_field[y0:y1, x0:x1][tm > 0])
            fx = math.cos(th + math.pi / 2)
            fy = math.sin(th + math.pi / 2)
            ts = np.arange(-patch_px // 2, patch_px // 2)
            xs = np.clip((cx + ts * fx).astype(int), 0, W - 1)
            ys = np.clip((cy + ts * fy).astype(int), 0, H - 1)
            trace = sharp[ys, xs].astype(np.float32)
            trace -= trace.mean()
            if trace.std() < 1e-3:
                continue
            spec = np.abs(rfft(trace))
            freqs = rfftfreq(len(trace), d=um_per_px)
            band = (freqs >= lo) & (freqs <= hi)
            total = float((spec[1:] ** 2).sum())
            if not band.any() or total <= 0:
                continue
            numer += n_zpix * float((spec[band] ** 2).sum())
            denom += n_zpix * total
            n_patches += 1
    if denom <= 0 or n_patches == 0:
        return float('nan')
    return float(numer / denom)

def paper_metrics(qt):
    """Consumes a `quantify_tissue` result. Returns the 5 published
    metrics plus enough metadata to reproduce the figure."""
    sharp = qt["sharpened"]
    clean = qt["clean_zdiscs"]
    tissue_mask = qt["tissue_mask"]
    z_discs = qt["z_discs"]
    um_per_px = qt["um_per_px"]

    OOP, _ = _oop_from_zdiscs(z_discs)
    gamma = _gamma_zdisks_over_bodies(z_discs)
    wOOP = OOP * gamma if (np.isfinite(OOP) and np.isfinite(gamma)) else float('nan')

    SL = _sarcomere_length_um_fft(sharp, tissue_mask, um_per_px)
    theta, _ = _local_orientation_field(sharp, tissue_mask, sigma_px=25.0)
    SPD = _spd_along_fibers_v2(sharp, clean, tissue_mask, theta, SL, um_per_px)

    dens = 1000.0 * qt["n_zdiscs"] / max(qt["tissue_area_um2"], 1.0)
    return {
        "sarcomere_length_um":       SL,
        "oop":                       OOP,
        "gamma":                     gamma,
        "weighted_oop":              wOOP,
        "spd":                       SPD,
        "zdisc_density_per_1000um2": dens,
        "n_zdiscs":                  qt["n_zdiscs"],
        "tissue_area_um2":           qt["tissue_area_um2"],
    }

print("metrics module ready: paper_metrics(quantify_tissue_result)")

In [ ]:
# ============================================================
# CONFIGS — edit these paths to point at your data
# ============================================================
# Two supported directory layouts:
#   (a) one folder per timepoint          → set `folders` = {tp: dir_path}
#   (b) all files in a single flat folder → set `folders` = {"__flat__": dir_path}
#       and provide `tp_regex` = {tp: pattern} to sort filenames by timepoint.
#
# Tissue ID extraction: `build_index` first STRIPS the timepoint / stain
# tokens (see `DEFAULT_ID_STRIP` below) then takes the trailing digit block
# from what remains. This handles filenames like
#   Quartz_ET1_007_baseline.nd2      → 007
#   quartz_mava_4hour_if_009.nd2     → 009
# You can override the strip pattern per-config via `id_strip_regex`.
#
# `reps` names the two tissues shown in Panel A (rep_1 = top row, rep_2
# = bottom row of the image grid).
# ============================================================

# Default: strip common timepoint tokens + "if" stain tag before
# extracting the trailing digit block as tissue ID.
DEFAULT_ID_STRIP = (r'(?:^|[_\-. ])'
                    r'(baseline|base|pre'
                    r'|4hours?|4hrs?|4h'
                    r'|8hours?|8hrs?|8h'
                    r'|7\.5hours?|7\.5hrs?|7\.5h|7\.5|7p5h|7_5h'
                    r'|24hours?|24hrs?|24h'
                    r'|if)(?=[_\-. ]|\.|$)')

CONFIGS = {
    "et1": {
        "label":       "ET-1",
        "timepoints":  ["baseline", "4h", "8h"],
        "folders": {
            "baseline": "/Users/ersm/Desktop/ET1 Images to analyse/baseline",
            "4h":       "/Users/ersm/Desktop/ET1 Images to analyse/4h",
            "8h":       "/Users/ersm/Desktop/ET1 Images to analyse/7.5h",
        },
        "file_glob":    "*.nd2",
        "reps":         {"rep_1": "008", "rep_2": "007"},
        "um_per_px":    0.16,
        "scale_bar_um": 50,
    },
    "mava": {
        "label":       "Mavacamten",
        "timepoints":  ["baseline", "4h", "8h"],
        "folders":     {"__flat__": "/Users/ersm/Desktop/ai_coding/Files to use_Ca_Mava"},
        "tp_regex": {
            "baseline": r"baseline",
            "4h":       r"(?:^|[^A-Za-z0-9])(4hours?|4hrs?|4h)(?=[^A-Za-z0-9]|$)",
            "8h":       r"(?:^|[^A-Za-z0-9])(8hours?|8hrs?|8h)(?=[^A-Za-z0-9]|$)",
        },
        "file_glob":    "*.nd2",
        "reps":         {"rep_1": "009", "rep_2": "020"},
        "um_per_px":    0.16,
        "scale_bar_um": 50,
    },
}

for name, cfg in CONFIGS.items():
    print(f"CONFIGS[{name!r}]: {cfg['label']}, {len(cfg['timepoints'])} timepoints, "
          f"um/px={cfg['um_per_px']}, reps={cfg['reps']}")


In [ ]:
# ============================================================
# BATCH RUNNER — discover files, process in parallel, write CSV
# ============================================================

def _extract_tissue_id(filename, id_strip_regex=None):
    """Strip timepoint/stain tokens, then return the trailing digit block."""
    stem = filename.rsplit('.', 1)[0]
    strip_pat = re.compile(id_strip_regex or DEFAULT_ID_STRIP, re.IGNORECASE)
    stripped = strip_pat.sub('_', stem)
    m = re.search(r'(\d{2,5})(?!.*\d)', stripped)
    return m.group(1) if m else None

def build_index(cfg):
    """Return {tid: {tp: Path}} of files matching the config."""
    tp_pattern_by_tp = {}
    if "__flat__" in cfg["folders"]:
        assert "tp_regex" in cfg, "__flat__ folders require 'tp_regex'"
        for tp, pat in cfg["tp_regex"].items():
            tp_pattern_by_tp[tp] = re.compile(pat, re.IGNORECASE)
    id_strip = cfg.get("id_strip_regex")

    def _which_tp(fname):
        if tp_pattern_by_tp:
            hits = [tp for tp, pat in tp_pattern_by_tp.items() if pat.search(fname)]
            return hits[0] if len(hits) == 1 else None
        return None

    index = defaultdict(dict)
    if "__flat__" in cfg["folders"]:
        d = Path(cfg["folders"]["__flat__"])
        for p in sorted(d.glob(cfg["file_glob"])):
            tp = _which_tp(p.name)
            if tp is None:
                continue
            tid = _extract_tissue_id(p.name, id_strip)
            if not tid:
                continue
            index[tid][tp] = p
    else:
        for tp in cfg["timepoints"]:
            d = Path(cfg["folders"][tp])
            for p in sorted(d.glob(cfg["file_glob"])):
                tid = _extract_tissue_id(p.name, id_strip)
                if not tid:
                    continue
                index[tid][tp] = p
    return dict(index)

def run_dataset(cfg, name, max_workers=4, verbose=True):
    """Discover files, run detector + paper_metrics on every image,
    return (complete_tids, results, rep_qts).

    - complete_tids: list[str] — tissues that had files for every timepoint
    - results:       {(tid, tp): paper_metrics_dict}
    - rep_qts:       {(tid, tp): quantify_tissue_dict} for the two rep tissues only

    Also writes results/pm_<name>.csv.
    """
    idx = build_index(cfg)
    complete_tids = sorted(
        [t for t, tps in idx.items()
         if all(tp in tps for tp in cfg["timepoints"])],
        key=lambda x: int(x))
    partial = sorted(set(idx.keys()) - set(complete_tids), key=lambda x: int(x))
    if verbose:
        print(f"[{name}] discovered {len(idx)} tissues, "
              f"{len(complete_tids)} complete → {complete_tids}")
        if partial:
            for t in partial:
                have = list(idx[t].keys())
                miss = [tp for tp in cfg["timepoints"] if tp not in have]
                print(f"  partial #{t}: have {have}, missing {miss}")

    rep_tids = set(cfg["reps"].values())
    for rep_id in rep_tids:
        if rep_id not in complete_tids:
            raise ValueError(
                f"Representative tissue #{rep_id} is not in complete_tids "
                f"({complete_tids}). Fix `CONFIGS[{name!r}]['reps']`.")

    work = [(tid, tp, idx[tid][tp])
            for tid in complete_tids for tp in cfg["timepoints"]]
    if verbose:
        print(f"[{name}] processing {len(work)} images with {max_workers} workers...")

    def _run(item):
        tid, tp, path = item
        qt = quantify_tissue(path, cfg["um_per_px"])
        pm = paper_metrics(qt)
        # keep the heavy `qt` only for rep tissues (needed by Panel A)
        keep_qt = qt if tid in rep_tids else None
        return tid, tp, pm, keep_qt

    results = {}
    rep_qts = {}
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(_run, it): it for it in work}
        for fut in as_completed(futs):
            tid, tp, pm, qt = fut.result()
            results[(tid, tp)] = pm
            if qt is not None:
                rep_qts[(tid, tp)] = qt
            if verbose:
                print(f"  [{tid}/{tp}]  spd={pm['spd']:.3f}  γ={pm['gamma']:.3f}  "
                      f"n_zd={pm['n_zdiscs']}  dens={pm['zdisc_density_per_1000um2']:.1f}")
    if verbose:
        print(f"[{name}] done in {time.perf_counter()-t0:.1f}s wall")

    # per-image CSV
    csv_path = RESULT_DIR / f"pm_{name}.csv"
    metric_cols = ["sarcomere_length_um", "oop", "gamma", "weighted_oop", "spd",
                   "zdisc_density_per_1000um2", "n_zdiscs", "tissue_area_um2"]
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["tissue_id", "timepoint"] + metric_cols)
        for tid in complete_tids:
            for tp in cfg["timepoints"]:
                pm = results[(tid, tp)]
                w.writerow([tid, tp] + [pm[c] for c in metric_cols])
    if verbose:
        print(f"[{name}] wrote {csv_path}")

    return complete_tids, results, rep_qts

print("batch runner ready: run_dataset(cfg, name)")

In [ ]:
# ============================================================
# FIGURE — 2×3 image grid on top, two paired-dot panels below
# ============================================================

def _pair_stats(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    diff = b - a
    n = a.size
    if n < 2 or np.allclose(diff, 0):
        return dict(n=n, p_wilcoxon=1.0, dz=0.0, mean_diff=float(diff.mean()))
    try:
        w = sps.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
        p = float(w.pvalue)
    except ValueError:
        p = float('nan')
    sd = diff.std(ddof=1)
    dz = float(diff.mean() / sd) if sd > 0 else float('nan')
    return dict(n=n, p_wilcoxon=p, dz=dz, mean_diff=float(diff.mean()))

def _p_star(p):
    if not np.isfinite(p): return "n/a"
    if p < 0.001: return "***"
    if p < 0.01 : return "**"
    if p < 0.05 : return "*"
    return "n.s."

def _to_gray_rgb(sharp):
    x = sharp.astype(np.float32)
    lo, hi = np.percentile(x, (1, 99))
    x = np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)
    x8 = (x * 255).astype(np.uint8)
    return np.stack([x8, x8, x8], axis=-1)

def _scale_bar(ax, img_shape, um, um_per_px, color='white', pad_frac=0.05):
    H, W = img_shape[:2]
    bar_px = um / um_per_px
    y = H - int(pad_frac * H) - 4
    x1 = W - int(pad_frac * W)
    x0 = x1 - bar_px
    ax.plot([x0, x1], [y, y], color=color, linewidth=2.5, solid_capstyle='butt')
    ax.text((x0 + x1) / 2, y - 8, f"{um} μm",
            color=color, ha='center', va='bottom',
            fontsize=8, fontweight='bold')

def _spaghetti(ax, data_by_tp, tps, tp_labels, tp_colors, title, ylabel, n_paired):
    x = np.arange(len(tps))
    for i in range(n_paired):
        ys = [data_by_tp[tp][i] for tp in tps]
        ax.plot(x, ys, 'o-', color='0.55', alpha=0.55, linewidth=1.0,
                markersize=5, zorder=1)
    for j, tp in enumerate(tps):
        arr = data_by_tp[tp]
        ax.scatter([x[j]] * len(arr), arr, s=36, color=tp_colors[j],
                   edgecolor='white', linewidth=0.7, zorder=2)
    means = [data_by_tp[tp].mean() for tp in tps]
    sems  = [data_by_tp[tp].std(ddof=1) / math.sqrt(n_paired)
             if n_paired > 1 else 0 for tp in tps]
    ax.errorbar(x + 0.22, means, yerr=sems, fmt='s-', color='black',
                markersize=7, linewidth=2, capsize=4, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(tp_labels)
    ax.set_xlim(-0.4, len(tps) - 0.4 + 0.4)
    all_vals = np.concatenate([data_by_tp[tp] for tp in tps])
    lo, hi = float(all_vals.min()), float(all_vals.max())
    span = hi - lo if hi > lo else max(abs(hi), 1e-6)
    ax.set_ylim(lo - 0.35 * span, hi + 0.15 * span)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, axis='y', alpha=0.25)

def _annotate_pairs(ax, data_by_tp, tps, fmt="+.4f"):
    lines = []
    for i in range(len(tps)):
        for j in range(i + 1, len(tps)):
            s = _pair_stats(data_by_tp[tps[i]], data_by_tp[tps[j]])
            lines.append(f"{tps[i]}→{tps[j]}: "
                         f"Δ={s['mean_diff']:{fmt}}  "
                         f"p={s['p_wilcoxon']:.3f} {_p_star(s['p_wilcoxon'])}")
    ax.text(0.02, 0.03, "\n".join(lines), transform=ax.transAxes,
            fontsize=9, va='bottom', ha='left', family='monospace',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='0.7', alpha=0.92))

def make_figure(results, cfg, name, complete_tids, rep_qts):
    """Render the 2×3 image grid + SPD + γ paired-dot figure.
    Writes <FIG_DIR>/<name>.{png,svg,tiff}. Returns the paired-stats dict."""
    tps = cfg["timepoints"]
    rep_1 = cfg["reps"]["rep_1"]
    rep_2 = cfg["reps"]["rep_2"]
    label = cfg["label"]
    um_per_px = cfg["um_per_px"]
    scale_um = cfg["scale_bar_um"]

    tp_labels_disp = [f"Baseline" if tp == "baseline" else f"{label} {tp}" for tp in tps]
    tp_colors = ["#4C72B0", "#DD8452", "#C44E52"][: len(tps)]

    # Paired arrays for the two metrics we ship
    def _arr(key):
        return {tp: np.array([results[(t, tp)][key] for t in complete_tids],
                             dtype=float)
                for tp in tps}
    spd_a   = _arr("spd")
    gamma_a = _arr("gamma")

    print(f"[{name}] paired stats (n = {len(complete_tids)}):")
    for arr_name, arr in [("SPD", spd_a), ("γ", gamma_a)]:
        print(f"  {arr_name}")
        for i in range(len(tps)):
            for j in range(i + 1, len(tps)):
                s = _pair_stats(arr[tps[i]], arr[tps[j]])
                print(f"    {tps[i]}→{tps[j]}: Δ={s['mean_diff']:+.4f}  "
                      f"dz={s['dz']:+.2f}  p={s['p_wilcoxon']:.4f}  "
                      f"{_p_star(s['p_wilcoxon'])}")

    fig = plt.figure(figsize=(15, 8.5))
    gs_outer = GridSpec(2, 1, figure=fig,
                        height_ratios=[1.0, 1.15],
                        hspace=0.28)
    gs_A = gs_outer[0, 0].subgridspec(2, len(tps), hspace=0.05, wspace=0.03)

    axA_axes = []
    for row_i, tid in enumerate([rep_1, rep_2]):
        for col, tp in enumerate(tps):
            ax = fig.add_subplot(gs_A[row_i, col])
            axA_axes.append(ax)
            qt = rep_qts.get((tid, tp))
            if qt is None:
                raise KeyError(f"rep_qts missing ({tid}, {tp}) — did run_dataset finish?")
            ax.imshow(_to_gray_rgb(qt["sharpened"]))
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values(): s.set_visible(False)
            if row_i == 0:
                ax.set_title(tp_labels_disp[col], fontsize=12, fontweight='bold')
            if col == 0:
                _scale_bar(ax, qt["sharpened"].shape, scale_um, um_per_px)
    axA_axes[0].set_ylabel(f"Tissue #{rep_1}", fontsize=11,
                            fontweight='bold', labelpad=8)
    axA_axes[len(tps)].set_ylabel(f"Tissue #{rep_2}", fontsize=11,
                                    fontweight='bold', labelpad=8)

    gs_bot = gs_outer[1, 0].subgridspec(1, 2, wspace=0.30)
    axB = fig.add_subplot(gs_bot[0, 0])
    axC = fig.add_subplot(gs_bot[0, 1])
    _spaghetti(axB, spd_a, tps, tp_labels_disp, tp_colors,
               "Sarcomeric packing density (SPD)",
               "SPD  (Z-disc mass on regular lattice)",
               len(complete_tids))
    _spaghetti(axC, gamma_a, tps, tp_labels_disp, tp_colors,
               "γ — Z-disk fraction",
               "γ  (elongated Z-disks / all foreground)",
               len(complete_tids))
    _annotate_pairs(axB, spd_a,   tps)
    _annotate_pairs(axC, gamma_a, tps)

    axA_axes[0].text(-0.10, 1.28, "A", transform=axA_axes[0].transAxes,
                     fontsize=22, fontweight='bold', ha='right', va='top')
    axB.text(-0.13, 1.06, "B", transform=axB.transAxes,
             fontsize=22, fontweight='bold', ha='right', va='top')
    axC.text(-0.13, 1.06, "C", transform=axC.transAxes,
             fontsize=22, fontweight='bold', ha='right', va='top')

    fig.suptitle(f"{label} does not disassemble sarcomeres  "
                 f"(n = {len(complete_tids)} paired tissues)",
                 fontsize=14, fontweight='bold', y=0.995)
    plt.subplots_adjust(left=0.06, right=0.985, top=0.955, bottom=0.06)

    stem = FIG_DIR / name
    for ext in ('png', 'svg', 'tiff'):
        fig.savefig(str(stem) + f".{ext}", dpi=300, bbox_inches='tight')
        print(f"  saved → {stem}.{ext}")
    plt.show()
    return dict(spd=spd_a, gamma=gamma_a)

print("figure module ready: make_figure(results, cfg, name, complete_tids, rep_qts)")

In [ ]:
# ============================================================
# RUN — ET-1 dataset
# ============================================================
cfg = CONFIGS["et1"]
complete_tids, results, rep_qts = run_dataset(cfg, name="et1")
arrays = make_figure(results, cfg, name="et1",
                     complete_tids=complete_tids,
                     rep_qts=rep_qts)

In [ ]:
# ============================================================
# RUN — Mavacamten dataset
# ============================================================
cfg = CONFIGS["mava"]
complete_tids, results, rep_qts = run_dataset(cfg, name="mava")
arrays = make_figure(results, cfg, name="mava",
                     complete_tids=complete_tids,
                     rep_qts=rep_qts)